# 03 — Silver | Player Game Stats and Game Details from Boxscores (incremental)

**Project:** MLB Performance Intelligence Platform

Parses each boxscore (one per completed game) into player-level facts. The API returns the
players of each team as a map keyed by `"ID<playerId>"`; it is parsed with a typed `from_json`
schema and exploded with Spark, so thousands of games are processed in parallel without UDFs.

| Table | Grain | Key |
|---|---|---|
| `silver.fact_batting_game` | one row per player and team who batted, ran or fielded in the lineup | `game_pk, player_id, team_id` |
| `silver.fact_pitching_game` | one row per pitcher appearance | `game_pk, player_id, team_id` |
| `silver.game_details` | one row per game: attendance, duration, weather, home-plate umpire | `game_pk` |

**Rules:** a batting row requires a `battingOrder` (starters end in `00`, substitutes `01`–`09`);
innings pitched are stored as **outs** (exact) and converted to decimal innings (outs / 3);
quality start = starter with ≥ 18 outs and ≤ 3 earned runs.

**Why `team_id` is part of the key:** on 26 Aug 2024 Danny Jansen played for *both* teams in the
same game (a suspended Blue Jays–Red Sox game resumed after his trade to Boston). A player-game
key without the team would collapse his two lines; the data-quality check DQ-005 caught it.

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
import schemas as S
for _m in (C, P, S):
    importlib.reload(_m)
from pyspark.sql import functions as F

dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"], "Reprocess all Bronze history")
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"
RUN_ID = P.new_run_id()
CONSUMER = "silver_boxscores"

if FULL_REFRESH:  # rebuild the tables from all Bronze history
    for t in ("fact_batting_game", "fact_pitching_game", "game_details"):
        spark.sql(f"DROP TABLE IF EXISTS {C.table('silver', t)}")
wm = None if FULL_REFRESH else P.get_watermark(spark, CONSUMER)
src = P.latest_payloads(spark, "boxscore", wm)
new_wm = P.max_ingested(src)

box = src.select(F.col("request_key").cast("bigint").alias("game_pk"),
                 F.from_json("payload", S.BOXSCORE).alias("b"))
print(f"Watermark: {wm} -> {new_wm} | boxscores to process: {box.count()}")

# Game context (date, season, type) comes from fact_game, built by the previous task
ctx = spark.table(C.table("silver", "fact_game")).select("game_pk", "season", "game_type", "season_type", "official_date")

## Explode players of both teams

In [ ]:
def team_players(me, opp, is_home):
    return box.select(
        "game_pk",
        F.col(f"b.teams.{me}.team.id").alias("team_id"),
        F.col(f"b.teams.{opp}.team.id").alias("opponent_id"),
        F.lit(is_home).alias("is_home"),
        F.col(f"b.teams.{me}.pitchers").alias("pitcher_order"),
        F.explode(F.map_values(f"b.teams.{me}.players")).alias("p"),
    )

players = team_players("home", "away", True).unionByName(team_players("away", "home", False)) \
    .join(ctx, "game_pk", "inner")

## fact_batting_game

In [ ]:
b = "p.stats.batting"
batting = (players.where(F.col("p.battingOrder").isNotNull()).select(
    "game_pk", F.col("p.person.id").alias("player_id"), F.col("p.person.fullName").alias("player_name"),
    "team_id", "opponent_id", "is_home", "season", "game_type", "season_type", "official_date",
    F.col("p.battingOrder").cast("int").alias("batting_order"),
    (F.col("p.battingOrder").cast("int") / 100).cast("int").alias("lineup_slot"),
    (F.col("p.battingOrder").cast("int") % 100 == 0).alias("is_starter"),
    F.col("p.position.abbreviation").alias("position"),
    *[F.coalesce(F.col(f"{b}.{src}"), F.lit(0)).alias(dst) for src, dst in [
        ("plateAppearances", "pa"), ("atBats", "ab"), ("runs", "r"), ("hits", "h"), ("doubles", "doubles"),
        ("triples", "triples"), ("homeRuns", "hr"), ("rbi", "rbi"), ("baseOnBalls", "bb"),
        ("intentionalWalks", "ibb"), ("strikeOuts", "so"), ("hitByPitch", "hbp"), ("stolenBases", "sb"),
        ("caughtStealing", "cs"), ("sacFlies", "sf"), ("sacBunts", "sh"), ("totalBases", "tb"),
        ("leftOnBase", "lob"), ("groundIntoDoublePlay", "gidp")]],
).withColumn("singles", F.col("h") - F.col("doubles") - F.col("triples") - F.col("hr"))
 .withColumn("updated_at", F.current_timestamp()))

with P.log_step(spark, RUN_ID, "silver", "fact_batting_game", str(new_wm)) as step:
    step.records = P.merge_upsert(spark, batting, C.table("silver", "fact_batting_game"), ["game_pk", "player_id", "team_id"],
        comment="Silver: one batting line per player per game (lineup slot, PA, AB, H, 2B, 3B, HR, RBI, BB, SO, SB...)")

## fact_pitching_game

In [ ]:
pi = "p.stats.pitching"
outs = F.coalesce(F.col(f"{pi}.outs"), F.lit(0))
er = F.coalesce(F.col(f"{pi}.earnedRuns"), F.lit(0))
starter = F.coalesce(F.col(f"{pi}.gamesStarted"), F.lit(0)) == 1
pitching = (players.where(F.col(f"{pi}.battersFaced").isNotNull() | F.col(f"{pi}.outs").isNotNull()).select(
    "game_pk", F.col("p.person.id").alias("player_id"), F.col("p.person.fullName").alias("player_name"),
    "team_id", "opponent_id", "is_home", "season", "game_type", "season_type", "official_date",
    F.expr("array_position(pitcher_order, p.person.id)").cast("int").alias("appearance_order"),
    starter.alias("is_starter"),
    outs.alias("outs"),
    F.round(outs / 3, 4).alias("innings_pitched"),
    F.format_string("%d.%d", (outs / 3).cast("int"), outs % 3).alias("innings_pitched_display"),
    *[F.coalesce(F.col(f"{pi}.{src}"), F.lit(0)).alias(dst) for src, dst in [
        ("battersFaced", "batters_faced"), ("numberOfPitches", "pitches"), ("strikes", "strikes"),
        ("hits", "h"), ("runs", "r"), ("earnedRuns", "er"), ("baseOnBalls", "bb"), ("intentionalWalks", "ibb"),
        ("strikeOuts", "so"), ("homeRuns", "hr"), ("hitBatsmen", "hbp"), ("wildPitches", "wp"),
        ("wins", "w"), ("losses", "l"), ("saves", "sv"), ("saveOpportunities", "svo"), ("holds", "hld"),
        ("blownSaves", "bs"), ("completeGames", "cg"), ("shutouts", "sho"),
        ("inheritedRunners", "inherited_runners"), ("inheritedRunnersScored", "inherited_runners_scored")]],
    (starter & (outs >= 18) & (er <= 3)).alias("is_quality_start"),
).withColumn("decision", F.when(F.col("w") == 1, "W").when(F.col("l") == 1, "L")
                          .when(F.col("sv") == 1, "SV").when(F.col("hld") == 1, "HLD"))
 .withColumn("updated_at", F.current_timestamp()))

with P.log_step(spark, RUN_ID, "silver", "fact_pitching_game", str(new_wm)) as step:
    step.records = P.merge_upsert(spark, pitching, C.table("silver", "fact_pitching_game"), ["game_pk", "player_id", "team_id"],
        comment="Silver: one pitching line per pitcher appearance (outs, IP, H, ER, BB, SO, HR, pitches, decision, quality start)")

## game_details (from the boxscore `info` block)

In [ ]:
def info(label):
    v = F.get(F.filter("b.info", lambda x: x.label == label), 0).getField("value")  # get(): NULL if absent
    return F.regexp_replace(v, r"\.$", "")

dur = info("T")
details = box.select(
    "game_pk",
    P.num(F.regexp_replace(info("Att"), r"[^0-9]", "")).alias("attendance"),
    (P.num(F.regexp_extract(dur, r"^(\d+):(\d+)", 1)) * 60
     + P.num(F.regexp_extract(dur, r"^(\d+):(\d+)", 2))).alias("duration_minutes"),
    info("Weather").alias("weather"),
    info("Wind").alias("wind"),
    info("First pitch").alias("first_pitch_local"),
    F.get(F.filter("b.officials", lambda o: o.officialType == "Home Plate"), 0).getField("official").getField("fullName").alias("home_plate_umpire"),
).withColumn("updated_at", F.current_timestamp())

with P.log_step(spark, RUN_ID, "silver", "game_details", str(new_wm)) as step:
    step.records = P.merge_upsert(spark, details, C.table("silver", "game_details"), ["game_pk"],
        comment="Silver: attendance, duration, weather and home-plate umpire per completed game")

P.set_watermark(spark, CONSUMER, new_wm)

In [ ]:
display(spark.sql(f"""
    SELECT season, season_type, COUNT(DISTINCT game_pk) AS games, COUNT(DISTINCT player_id) AS batters,
           SUM(hr) AS home_runs, ROUND(SUM(h) / NULLIF(SUM(ab), 0), 3) AS league_avg
    FROM {C.table('silver', 'fact_batting_game')} GROUP BY season, season_type ORDER BY season, season_type"""))